# 01 — Inspeksi dataset

Tujuan: memahami dataset SEBELUM training — sumber & lisensi, ukuran/orientasi gambar, distribusi kelas,
ukuran objek, jumlah objek per foto, dan contoh visual dengan bounding box.

Prasyarat (sekali, dari folder `ml/`):
```powershell
.venv\Scripts\python -m wallacea_ml datasets sample uavmulticrop UAVmulticrop.zip --pattern "*/maize/*" --limit 200 --max-mb 400
.venv\Scripts\python -m wallacea_ml prepare configs\prepare\uavmulticrop_maize.yaml
```
Kernel: **Python (wallacea-ml)**.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## Katalog dataset (diperiksa ke sumber asli)

In [ ]:
from wallacea_ml.datasets import load_catalog
cat = load_catalog()
display(_df([{"id": d["id"], "status": d["status"], "tanaman": ", ".join(d["crops"]),
              "lisensi dataset": d["license"]["dataset"], "lisensi kode": d["license"]["code"],
              "bobot": d["license"]["weights"], "format": d["annotation_format"][:60]} for d in cat.values()]))

## Pilih dataset yang sudah di-`prepare`

In [ ]:
DATASET = "uavmulticrop_maize"          # nama folder di data/prepared/
ROOT = ML_ROOT / "data" / "prepared" / DATASET
from wallacea_ml.manifest import load_manifest
from wallacea_ml.inspection import inspect_manifest
m = load_manifest(ROOT / "manifest.json")
assert not m.errors, m.errors
st = inspect_manifest(m, ROOT)
summary = st.summary()
print(json.dumps(summary, indent=1, default=str))

## Gambar per grup (lahan/penerbangan) — dasar split

In [ ]:
from collections import Counter
display(_df([{"field_id": f, "images": n, "boxes": sum(r["boxes"] for r in st.images if r["field_id"] == f)}
             for f, n in sorted(Counter(r["field_id"] for r in st.images).items())]))

## Distribusi ukuran gambar, objek per foto, dan ukuran objek

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist([r["boxes"] for r in st.images], bins=20); ax[0].set_title("objek per foto")
ax[1].hist([b["w_px"] for b in st.boxes], bins=40); ax[1].set_title("lebar box (px, gambar terorientasi)")
ax[2].hist([b["w"] * b["h"] for b in st.boxes], bins=40, log=True); ax[2].set_title("luas box relatif (log)")
plt.tight_layout(); plt.show()
print("ukuran gambar:", summary["image_sizes"], "| orientasi:", summary["orientation"])
print("Catatan: API memprediksi FOTO PENUH di-letterbox ke WALLACEA_IMGSZ. Objek sangat kecil relatif terhadap foto akan hilang saat diperkecil.")

## Contoh visual (piksel terorientasi EXIF, sama dengan editor WALLACEA)

In [ ]:
from wallacea_ml import backend
from wallacea_ml.inspection import read_yolo_labels
from wallacea_ml.viz import draw_boxes
names = {i: lab for lab, i in m.label_map.items()}
sample = [e for e in m.images if e.labels][:: max(1, len(m.labeled) // 4)][:4]
fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 4))
for ax, e in zip(axes if len(sample) > 1 else [axes], sample):
    boxes, _ = read_yolo_labels(ROOT / e.labels, len(names))
    ax.imshow(draw_boxes(backend.load_oriented(ROOT / e.image), [(names[c], bb) for c, bb in boxes]))
    ax.set_title(f"{e.field_id} · {len(boxes)} box"); ax.axis("off")
plt.tight_layout(); plt.show()